# Chapter 4 — Model Evaluation and Explainability

This notebook loads the pre-computed evaluation report and produces all
dissertation figures.  No model is re-loaded here; everything is read from
`reports/evaluation_report.json` so the notebook runs quickly even on
machines without a GPU.

**Run `python -m src.evaluation.evaluator` first** to generate the report.

Sections:
1. Load report and print dataset info
2. Per-model metrics table
3. ROC curves on one chart
4. Precision-recall curves
5. Confusion matrices side by side
6. SHAP feature importance bar chart
7. SHAP waterfall plots
8. DistilBERT attention visualisation

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REPORT_PATH = PROJECT_ROOT / 'reports' / 'evaluation_report.json'
REPORTS_DIR = PROJECT_ROOT / 'reports'

plt.rcParams.update({
    'figure.dpi':         150,
    'font.family':        'sans-serif',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.3,
})

print('Project root:', PROJECT_ROOT)
print('Report file: ', REPORT_PATH)

## 1. Load report and dataset info

In [ ]:
with open(REPORT_PATH) as f:
    report = json.load(f)

info = report.get('dataset_info', {})
print('Test set overview')
print('-' * 40)
for k, v in info.items():
    print(f'  {k:<30} {v}')

## 2. Per-model metrics table

In [ ]:
MODELS     = ['xgboost', 'bert', 'fusion']
MODEL_LABELS = {'xgboost': 'XGBoost', 'bert': 'DistilBERT', 'fusion': 'Fusion (LightGBM)'}
METRIC_KEYS  = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']

rows = []
for model_name in MODELS:
    m = report.get(model_name, {})
    if not m or m.get('skipped'):
        continue
    row = {'Model': MODEL_LABELS.get(model_name, model_name)}
    for k in METRIC_KEYS:
        val = m.get(k)
        row[k.replace('_', ' ').title()] = f'{val:.4f}' if val is not None else 'N/A'
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index('Model')
display(
    metrics_df.style
    .set_caption('Model Performance on Held-out Test Set')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '13px'), ('font-weight', 'bold')]
    }])
    .highlight_max(color='#d4edda', axis=0)
)

## 3. ROC curves

In [ ]:
MODEL_COLORS = {'xgboost': '#e41a1c', 'bert': '#377eb8', 'fusion': '#4daf4a'}

fig, ax = plt.subplots(figsize=(7, 6))

for model_name in MODELS:
    m   = report.get(model_name, {})
    roc = m.get('roc_curve', {})
    if not roc:
        continue
    fpr = roc['fpr']
    tpr = roc['tpr']
    auc = m.get('roc_auc', 0.0)
    ax.plot(
        fpr, tpr,
        color=MODEL_COLORS[model_name],
        linewidth=2.0,
        label=f"{MODEL_LABELS[model_name]} (AUC = {auc:.3f})"
    )

ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
save_path = REPORTS_DIR / 'roc_curves.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {save_path}')

## 4. Precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for model_name in MODELS:
    m   = report.get(model_name, {})
    prc = m.get('pr_curve', {})
    if not prc:
        continue
    precision_vals = prc['precision']
    recall_vals    = prc['recall']
    ap             = m.get('avg_precision', 0.0)
    ax.plot(
        recall_vals, precision_vals,
        color=MODEL_COLORS[model_name],
        linewidth=2.0,
        label=f"{MODEL_LABELS[model_name]} (AP = {ap:.3f})"
    )

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
save_path = REPORTS_DIR / 'pr_curves.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {save_path}')

## 5. Confusion matrices

In [ ]:
available = [m for m in MODELS if m in report and 'confusion_matrix' in report[m]]
n_models  = len(available)

if n_models == 0:
    print('No confusion matrices found in the report.')
else:
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.5))
    if n_models == 1:
        axes = [axes]

    class_names = ['Legitimate', 'Phishing']

    for ax, model_name in zip(axes, available):
        cm      = np.array(report[model_name]['confusion_matrix'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        sns.heatmap(
            cm_norm,
            annot=True,
            fmt='.2%',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            ax=ax,
            linewidths=0.5,
            cbar=False,
        )

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(
                    j + 0.5, i + 0.72, f'n={cm[i, j]:,}',
                    ha='center', va='center', fontsize=8, color='black'
                )

        ax.set_title(MODEL_LABELS[model_name], fontsize=12, fontweight='bold')
        ax.set_ylabel('True label',      fontsize=10)
        ax.set_xlabel('Predicted label', fontsize=10)

    fig.suptitle(
        'Confusion Matrices (row-normalised)',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    save_path = REPORTS_DIR / 'confusion_matrices.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {save_path}')

## 6. SHAP feature importance (XGBoost)

In [ ]:
shap_data = report.get('shap_feature_importances', {})

if not shap_data:
    print('SHAP feature importances not found in the report.')
    print('Run:  python -m src.evaluation.evaluator')
else:
    shap_df = (
        pd.DataFrame(list(shap_data.items()), columns=['feature', 'mean_abs_shap'])
        .sort_values('mean_abs_shap', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

    n       = len(shap_df)
    bar_colors = plt.cm.Blues(np.linspace(0.4, 0.9, n))[::-1]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(
        shap_df['feature'][::-1],
        shap_df['mean_abs_shap'][::-1],
        color=bar_colors,
        edgecolor='white',
        linewidth=0.5,
    )
    ax.set_xlabel('Mean |SHAP value|', fontsize=12)
    ax.set_title(
        f'XGBoost Feature Importance — Top {n} Features (SHAP)',
        fontsize=13, fontweight='bold', pad=12
    )
    ax.tick_params(axis='y', labelsize=9)
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout()
    save_path = REPORTS_DIR / 'shap_importance_bar.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {save_path}')

    print('\nTop 10 most important features:')
    display(
        shap_df.head(10)
        .style
        .format({'mean_abs_shap': '{:.5f}'})
        .hide(axis='index')
    )

## 7. SHAP waterfall plots

These plots are generated automatically when you run
`python -m src.evaluation.evaluator` and saved to `reports/`.
This cell just loads and displays them.

In [ ]:
from IPython.display import Image, display as ipy_display

waterfall_files = {
    'SHAP waterfall — phishing example':   REPORTS_DIR / 'shap_waterfall_phishing.png',
    'SHAP waterfall — legitimate example': REPORTS_DIR / 'shap_waterfall_legitimate.png',
}

for title, path in waterfall_files.items():
    if path.exists():
        print(f'\n{title}')
        ipy_display(Image(filename=str(path), width=750))
    else:
        print(f'{title}: not found at {path}')
        print('Run python -m src.evaluation.evaluator to generate it.')

## 8. DistilBERT attention visualisation

These plots are also pre-generated by the evaluator.  If they are not yet
present, set `RUN_BERT_ATTENTION = True` in the second cell below to
generate them on the fly (requires a loaded BERT model and some time).

In [ ]:
attention_files = {
    'DistilBERT attention — phishing example':   REPORTS_DIR / 'bert_attention_phishing.png',
    'DistilBERT attention — legitimate example': REPORTS_DIR / 'bert_attention_legitimate.png',
}

for title, path in attention_files.items():
    if path.exists():
        print(f'\n{title}')
        ipy_display(Image(filename=str(path), width=750))
    else:
        print(f'{title}: not found at {path}')

In [ ]:
# Set this to True to regenerate attention plots without running the full evaluator.
# Requires the trained BERT model in models/bert_phishing/ and a GPU (or patience).
RUN_BERT_ATTENTION = False

if RUN_BERT_ATTENTION:
    from src.text_pipeline.bert_classifier import PhishingBERTClassifier
    from src.evaluation.evaluator import load_test_data
    from src.evaluation.explainability import BERTAttentionVisualizer

    MODELS_DIR = PROJECT_ROOT / 'models'
    bert_clf   = PhishingBERTClassifier(models_dir=MODELS_DIR).load('bert_phishing')

    df_test       = load_test_data()
    df_sample     = df_test.groupby('label', group_keys=False).apply(
        lambda g: g.sample(min(len(g), 2500), random_state=42)
    ).reset_index(drop=True)
    y_true_sample = df_sample['label'].values
    y_prob_sample = bert_clf.predict_proba(df_sample)[:, 1]

    viz   = BERTAttentionVisualizer(bert_clf, reports_dir=REPORTS_DIR)
    paths = viz.explain_both(df_sample, y_true_sample, y_prob_sample, bert_clf)

    for title, path in paths.items():
        print(f'{title}: {path}')
        ipy_display(Image(filename=path, width=750))

## Summary

All figures are saved to `reports/` and can be included directly in the
dissertation write-up:

| File | Description |
|---|---|
| `roc_curves.png` | ROC curves for all three models on the same chart |
| `pr_curves.png` | Precision-recall curves for all three models |
| `confusion_matrices.png` | Side-by-side confusion matrices (row-normalised) |
| `shap_importance_bar.png` | Top 20 XGBoost features by mean absolute SHAP value |
| `shap_waterfall_phishing.png` | SHAP waterfall — highest-confidence phishing sample |
| `shap_waterfall_legitimate.png` | SHAP waterfall — highest-confidence legitimate sample |
| `bert_attention_phishing.png` | DistilBERT token attention heatmap — phishing example |
| `bert_attention_legitimate.png` | DistilBERT token attention heatmap — legitimate example |